# Data Wrangling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ixnix/GE5280PythonGeosciences/blob/main/colab/module_8/8_DataWrangling.ipynb)

*Run this notebook on Google Colab — no setup required.*

# Module 8: Data Wrangling


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image

## Introduction  
In this module, we build on your foundational Pandas skills and explore more advanced techniques for manipulating and combining datasets. You will learn how to filter and sort data efficiently in order to uncover insights from large DataFrames. We will also cover concatenating and merging operations to combine multiple datasets into a single, coherent structure. By the end, you will be able to perform more complex data wrangling tasks and prepare datasets for in-depth analysis.  

## Learning Objectives  
- Apply advanced filtering and sorting techniques to Pandas DataFrames.  
- Combine datasets using concatenation and merging operations in Pandas.  

## Reading
- [Chapter 5](https://wesmckinney.com/book/pandas-basics)
- [Chapter 8](https://wesmckinney.com/book/data-wrangling)

## Locating and Editing Data

Today we are going to learn about filtering pandas DataFrames. Pandas provides a powerful way to tease out useful information.

We will be looking at data on Holocene Eruptions from the Smithsonian Holocene Volcano Database: http://volcano.si.edu/list_volcano_holocene.cfm. This link will download an xml file, which Excel can read, but for Pandas to work on it, we can convert it to a CSV file (UTF-8 encoding) from with excel.

We will see how to filter these data to pull out interesting information on Holocene Eruptions. Let's read in this data and first look at its length. If you look in the file, you will see that the header is in the second line (not the first), so header=1 in the argument list, after the PATH to the file.

In [ ]:
EruptionData = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_8/data/GVP_Volcano_List_Holocene.csv',header=1)
print('Number of Eruptions:',len(EruptionData))
EruptionData.head()

Wow, that's a lot of Eruptions! However, the DataFrame has a lot of information we really aren't interested in. For example, there are many eruptions in this data for which the evidence isn't strong ('Evidence Uncertain'). You can verify this, by printing out the **Series** "Activity Evidence".



In [ ]:
EruptionData['Activity Evidence']

In the previous module, we learned how to filter a DataFrame by putting what we wanted in a conditional statement enclosed in square brackets. Remembering that the conditional for "equal to" is "==", we can retrieve all the rows that contain 'Eruption Observed' in the column 'Activity Evidence' like this:

In [ ]:
#notice the conditional '==' which means 'equals to'
ObservedEruptions = EruptionData[EruptionData['Activity Evidence']=='Eruption Observed']
ObservedEruptions.head()

**DataFrame.loc[]** allows for filtering of DataFrames by label(s) or a boolean array.

Allowed inputs are:
- A single label, e.g. 5 or 'a', (note that 5 is interpreted as a label of the index, and never as an integer position along the index).
- A list or array of labels, e.g. ['a', 'b', 'c'].
- A slice object with labels, e.g. 'a':'f'.
- A boolean array of the same length as the axis being sliced, e.g. [True, False, True].

In [ ]:
EruptionData.loc[EruptionData['Activity Evidence']=='Eruption Observed'].head()

This statement does exactly the same thing as the conditional. The syntax of a **.loc** statement might look trickier, but it will make your life easier as things get more complicated. It is computationally faster and has more tricks up its sleeve as we shall see soon.


Now let's look at some big eruptions we might be interested in. One of the most famous Volcanic eruptions is the 1980 Eruption of Mount St. Helens (Washington State). To find it, let's search for Holocene Eruptions of Mount St. Helens.

In [ ]:
EruptionData.loc[EruptionData['Volcano Name']=='St. Helens']

As we can see, simple conditional statements like this enable us to filter large datasets for the small amount of information we're interested in.

Although the above statement would work equally well without the **.loc** method, we can add some whistles and bells. The use of the **.loc** syntax allows us to search through a particular column (Series) by putting a comma after your conditional statement followed by another Series name. 

That is, **.loc** supports simultaneous row and column selection.

Say we want the 'Last Known Eruption' of all stratovolcanoes. We could do this:

In [ ]:
stratos = EruptionData.loc[EruptionData['Primary Volcano Type'].str.contains('Stratovolcano'),'Last Known Eruption']
stratos.head()

Here we used the syntax **DataFrame.Series.str.contains( )**. This allows us to get not only the type "Stratovolcano", but also "Stratovolcano(es)" and anything that has "Stratovolcano" in it.  

It is worth pointing out another way to accomplish the same thing using the method **isin()**. You can create a list of things you want (or don't want), then test if the string is in the list. Here is how it would work for this example to select things in the list:


In [ ]:
volcano_types = ['Stratovolcano','Stratovolcano(es)']
stratos = EruptionData[EruptionData['Primary Volcano Type'].isin(volcano_types)]
stratos.head()

And here's how it works if you don't want things in the list:

In [ ]:
volcano_types = ['Stratovolcano','Stratovolcano(es)']
not_stratos = EruptionData[EruptionData['Primary Volcano Type'].isin(volcano_types)==False]
not_stratos.head()

Now we can do stuff to this filtered DataFrame **stratos**. The **.loc** syntax also allows you to take a slice through the columns list to select a specific range of column headers:

In [ ]:
ColumnSlice = EruptionData.loc[EruptionData['Primary Volcano Type'].str.contains('Stratovolcano'), 'Volcano Name':'Last Known Eruption']
ColumnSlice.head()

Something else **.loc** can do is to change the values _inplace_ in DataFrames easily. Let's say we found a historical document that told us that the Unknown last eruption at Panarea was in 5000 BCE. We want to update the information in the **DataFrame** and can do it this way: 

In [ ]:
print('Before modifying:\n',EruptionData.loc[EruptionData['Volcano Name']=='Panarea','Last Known Eruption'])

EruptionData.loc[EruptionData['Volcano Name']=='Panarea','Last Known Eruption'] = '5000 BCE'

# and let's take a look: 
print('After modifying:\n',EruptionData.loc[EruptionData['Volcano Name']=='Panarea','Last Known Eruption'])

As we can see, the syntax for this can get complicated quickly, but we can retrieve and modify lots of data using a few lines of code. 

## Adding new columns

How can we add a new column based on existing data in the dataframe? Say, we would like to create a new column _NS_ in the data frame based on the volcano latitude:

In [ ]:
EruptionData['NS'] = np.nan
EruptionData.loc[ EruptionData.Latitude>0, 'NS'] = 'N'
EruptionData.loc[ EruptionData.Latitude<0, 'NS'] = 'S'

Or, we can use **lambda function** to add the new column in just one line of code:

In [ ]:
EruptionData['NS'] = EruptionData['Latitude'].map(lambda x: 'N' if x>0 else 'S')
EruptionData.head(5)

To verify the successful creation of the new column,

In [ ]:
EruptionData.columns

Filter the data frame based on the Elevation (m) and NS columns:

In [ ]:
EruptionData[(EruptionData['Elevation (m)'] > 3000) & (EruptionData['NS'] == 'N')]

## Sorting and Indexing

What if we wanted to sort our dataset so the most northerly eruptions come out on top? Pandas DataFrames have a method for this called **sort_values**. 

Normally, this will sort from lowest to highest (an "ascending" sort), but we can use the argument **ascending=False** to tell it to sort from highest to lowest.

In [ ]:
# First read in our DataFrame again:
EruptionData = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_8/data/GVP_Volcano_List_Holocene.csv',header=1)
NorthernToSouthern = EruptionData.sort_values(by='Latitude',ascending=False)
NorthernToSouthern.head()

Looks like the most northerly eruptions during the Holocene were at the East Gakkel Ridge and that the most northerly above sea-level eruption was on Jan Mayen in 1985.

Now let's try to get the first 10 rows in this DataFrame. We can do this using **.loc**, right?

In [ ]:
NorthernToSouthern.loc[0:10]

Oops! This didn't work as expected did it? Instead, we got the all the rows between the _indices_ of 0 and 10 which are not in any particular order now. When we sorted by Latitude, Pandas did not assign new indices and put the records in no particular order within a particular Latitude value. This is a "feature" of sorting functions. So... to get what we really wanted, which was the first 10 records in the NorthernToSouthern DataFrame, we can use the method **.iloc** instead of **.loc**.   

In [ ]:
NorthernToSouthern.iloc[0:10]

Much better. Now we can see that there were lots of eruptions during the Holocene on Iceland.  

But to solve our indexing problem with **.loc[]**, we can **reset** the index back to the order of 0 to the length of the Dataframe.

In [ ]:
# reset the indices to this list
# the drop=True drops the original index. Otherwise, it will be added into the DataFrame as a new column.
NorthernToSouthern = NorthernToSouthern.reset_index(drop=True)
NorthernToSouthern.head()

Another thing about indices: We can set the indices to one of the other column names, for example the "Volcano Name".  

In [ ]:
NorthernToSouthern = NorthernToSouthern.set_index('Volcano Name')
NorthernToSouthern.head()

## Sorting by awkward strings

In this volcanic eruption data set, the dates for the last known eruption are the dates in CE or BCE or unknown, so we cannot sort by that column header. But we can first drop all the rows where the 'Last Known Eruption' is 'unknown', then split the dates on the space, multiply all the dates with 'BCE' in them by -1 and sort by the resulting column.  

Let's do that step by step:

- to drop all the 'unknown' eruption dates, we can use the filtering statement:

In [ ]:
EruptionData = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_8/data/GVP_Volcano_List_Holocene.csv',header=1) # read the original data
KnownEruptionDates = EruptionData[ EruptionData['Last Known Eruption'].str.contains('unknown')==False ]

To see if this worked, we can look at all the unique eruption dates and see if 'unknown' is still there:

In [ ]:
KnownEruptionDates[KnownEruptionDates['Last Known Eruption'].str.contains('unknown')]

Nope. So now step 2: 

- To split a string on a particular key, in this case a space, we can use the syntax:

EruptionData\['Last Known Eruption'\].str.split()

In [ ]:
KnownEruptionDates['Last Known Eruption'].str.split().head()

Whoa! some of the entries are 'Unknown' and not 'unknown'. Sloppy! But we can handle that:

In [ ]:
KnownEruptionDates = KnownEruptionDates[KnownEruptionDates['Last Known Eruption'].str.contains('Unknown')==False]

The **.str.split()** method returns a list with the stuff before the split key (by default, empty space) as the first element and the stuff after the key is the second. 

We can make two arrays, one for the date and one for the 'CE' or 'BCE' tag. We first make an array (using the DataFrame method **.values**, transpose the array and assign the first row to a dataframe column named 'date' and the second row to a column named 'CE/BCE'.  

In [ ]:
dates = KnownEruptionDates['Last Known Eruption'].str.split(' ',expand=True).transpose()
print(dates.values)


In [ ]:
# put in the 'date' column:
KnownEruptionDates['date'] = dates.values[0].astype('int')

# put in the 'CE/BCE' column:
KnownEruptionDates['CE/BCE'] = dates.values[1].astype('str')

Because Pandas read in the 'Last Known Eruption' as a string, we need to convert the first part to an integer (that is the **.astype('int')** above). This means we can multiply the date in records with 'BCE' by -1.  


In [ ]:
KnownEruptionDates.loc[KnownEruptionDates['CE/BCE'].str.contains('BCE'),'date'] = -1*KnownEruptionDates['date']

# Now we can sort the Dataframe by the newly created 'date' column from the oldest to the newest eruptions
KnownEruptionDates.sort_values(by='date').head()

## Concatenation

We've been working on a data set that only had Holocene data in it, but the same Smithsonian website has a data set for Pleistocene volcanic eruptions too. We can concatenate both data sets into a single DataFrame (as long as they have the same columns) like this:  

In [ ]:
HoloceneEruptionData = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_8/data/GVP_Volcano_List_Holocene.csv',header=1)
HoloceneEruptionData.head()
#print(len(HoloceneEruptionData))

In [ ]:
PleistoceneEruptionData = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_8/data/GVP_Volcano_List_Pleistocene.csv',header=1) # read this in again
PleistoceneEruptionData.head()
#print(len(PleistoceneEruptionData))


To concatenate the Holocene and Pleistocene data into a single Dataframe:

In [ ]:
RecentEruptionData = pd.concat([HoloceneEruptionData,PleistoceneEruptionData])

# We can verify the results based on the Dataframe lengths
print(len(HoloceneEruptionData))
print(len(PleistoceneEruptionData))
print(len(RecentEruptionData))

In [ ]:
RecentEruptionData.head()

In [ ]:
RecentEruptionData.tail()

So a lot less is known about the Pleistocene eruptions than the Holocene ones.  

## Merge

There is another data set on the Smithsonian Website that is interesting. It has a list of all the currently active volcanoes. 

In [ ]:
ActiveVolcanoes = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_8/data/ActiveVolcanoes.csv')
ActiveVolcanoes.head()

There's a lot more information about each volcano in our HoloceneEruptionData DataFrame, which we could attach to the use Pandas **merge( )** method. There are many ways to use **merge**, but the idea is to identify what kind of join you want:  

In [ ]:
Image(url='https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_8/img/join-types.jpg',width=500)

We want to pair all the stuff from the Holocene volcanoes database with the ActiveVolcanoes data if it isn't already there ('Country' for example is in both).  

In [ ]:
MergedVolcanoes = ActiveVolcanoes.merge(KnownEruptionDates,how='left',left_on='Volcano',right_on='Volcano Name')
MergedVolcanoes.head()

You see that if a particular column (say 'Country') is in both DataFrames, the first one gets renamed 'Country_x' and the other one 'Country_y'. To clean that up we can rename the first to 'Country', then delete (drop) the second one out of the DataFrame. We also want to delete the column 'Volcano Name' because it is redundant.


In [ ]:
# inplace=True does the method 'in place' so we don't have to assign it to a new DataFrame
MergedVolcanoes.rename(columns={'Country_x':'Country'},inplace=True)
MergedVolcanoes.drop(['Country_y','Volcano Name'],axis=1,inplace=True)
MergedVolcanoes.head()

Looks like it worked.



## Using .unique() to find a list of categories, and string operations

Now that we have more information, we can start classifying these eruptions by type. For example, what tectonic settings are represented in this dataset? Pandas has a method called **.unique( )** that allows us to find all the unique values in a column.

In [ ]:
MergedVolcanoes['Tectonic Setting'].unique()

This tells us some useful information, including that some of the values are not a number (or 'nan'). 

We can get rid of these using the method **.dropna()**. While we are at it, we can delete the rows with no information on 'Volcanic Explosivity Index' data (Max VEI).

In [ ]:
MergedVolcanoes.dropna(subset=['Tectonic Setting'],inplace=True)
# The above is equivalent to MergedVolcanoes = MergedVolcanoes.dropna(subset=['Tectonic Setting']) without the inplace=True keyword

MergedVolcanoes.head()

In [ ]:
# Check if the rows with missing values (nan) of 'Tectonic Setting' are removed.
MergedVolcanoes['Tectonic Setting'].unique()